<div align="center">

# Road Lanes Detection

### Computer Vision Project 2025-2026  
**Author:** Emiliano Cacciuto  
**Student ID:** 2155347  

</div>

---

## Project Overview

**Topic:** Road Lanes Detection.

**Goal:** Find the street lines and Road Lanes starting to raw, real-world images.

**Scenario:** In this project, we will work with real images captured by a car riding on multi-lane streets. Most recent cars have ADAS systems to help the driver keep the car in the current lane and to alert him if the lane is changing. For these systems a key step is using **computer vision to detect the street lanes and in which lane the car is running**. 

---

## Libraries and Packages

The libraries and packages needed to run the project codes are the following ones

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
import urllib.request
import zipfile
import os
import typing as tp
import math
import pickle  # to store the parameters

---

## Dataset

The dataset of the project is composed by eight images: four "original-full" images showing real-world multi-lane streets and four "easier" version of them where the backgroung has been removed.

In particular the dataset need to be downloaded from https://lttm.dei.unipd.it/downloads/prj2025/roads.zip and we will extract the images in a local folder "Images" as follows

In [ ]:
# --- DOWNLOAD OF THE ZIP FILE ---
url = "https://lttm.dei.unipd.it/downloads/prj2025/roads.zip"
zip_file = "roads.zip"

if not os.path.exists(zip_file):  # download the file only if not already done previously
    urllib.request.urlretrieve(url, zip_file)
    print(f"{zip_file} has been downloaded")
else:
    print(f"{zip_file} is already present locally, download skipped")

# --- CREATE IMAGES FOLDER ---
images_folder = "Images"
os.makedirs(images_folder, exist_ok=True)

# --- EXTRACT THE IMAGES FROM ZIP FILE TO FOLDER "IMAGES" ---
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(images_folder)

print(f"All images have been extracted to the folder '{images_folder}'")

In [ ]:
# FUNCTION TO SHOW THE PAIR (FULL_IMAGE, EASIER IMAGE):
def show_imgs(img1, img2, title1="First Image", title2="Second Image"):
    
    plt.figure(figsize=(8, 4)) # Figure that shows each pair of images

    # --- FIRST IMAGE ---
    # Convert BGR -> RGB (for matplotlib from cv)
    img1 = cv.cvtColor(img1, cv.COLOR_BGR2RGB)

    plt.subplot(1, 2, 1) # (first pannel)
    plt.imshow(img1)
    if title1:
        plt.title(title1)
    plt.axis('off')

    # --- SECOND IMAGE ---
    # Convert BGR -> RGB as before
    img2= cv.cvtColor(img2, cv.COLOR_BGR2RGB)

    plt.subplot(1, 2, 2) # (second panel)
    plt.imshow(img2)
    if title2:
        plt.title(title2)
    plt.axis('off')
    

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# FUNCTION TO SHOW A SINGLE IMAGE 
def show_img(img, title=None):

    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    # Plot the image
    plt.figure() 
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# SHOW IMAGES PAIR
images_path= [
    ('Images/road1.png', 'Images/road1b.png'),
    ('Images/road2.jpg', 'Images/road2b.jpg'),
    ('Images/road3.jpg', 'Images/road3b.jpg'),
    ('Images/road4.png', 'Images/road4b.png')
]

for path1, path2 in images_path:
    image1 = cv.imread(path1)
    image2 = cv.imread(path2)
    show_imgs(image1, image2, title1='Original image', title2='Easier image')

In [ ]:
# SHOW A SINGLE IMAGE 
image = cv.imread("Images/road2.jpg")
show_img(image)

---

## Street lines and Road Detection [Simplified Images]

We will start with the task of **detecting the street lines and road lines** with the **"simplified" images**, namely those where the background has been removed. 

### Hough Transform to find the street lines

The first think we can do is to exploit the concept of the **Hough Transform** to detect street lines in image space $(x,y)$.

Let's recall that given the generic line equation
$$ y_i = ax_i+b$$

we can obtain the associated line in the parameter space $(a,b)$ as
$$ b = -x_ia+y_i$$

However, in order to include also vertical lines, we need to associat at each lines in the image space $(x,y)$ a sinusoidal curve in the polar (parameter) space $(\rho, \theta)$ following the relation
$$ \begin{gathered} x \cos \theta + y\sin \theta = \rho \\\Updownarrow \\y = -\frac{\cos \theta}{\sin \theta}x + \frac{\rho}{\sin \theta} \end{gathered} $$

The map from polar to cartesian can be made by the function:

In [ ]:
def polar2cartesian(radius: np.ndarray, angle: np.ndarray) -> tp.Tuple[np.ndarray]:
    return radius * np.array([np.cos(angle), np.sin(angle)]), np.array([np.sin(angle), -np.cos(angle)])

The steps of the Hough Transform can be summarized as follows:
* Execute an Edge Detection Algorithm to find the edges in the images;
* Quantize the parameter space $(\rho, \theta)$ in cells and associate each cell to a counter $A(\rho, \theta)$ called Accumulator;
* For each pixel $(x,y)$ of an edge:
  * Let $\theta$ vary in the quantized interval and compute the corresponding values $\rho = x \cos \theta +y\sin \theta$;
  * Increment the counter $A(\rho, \theta)$ for each crossed cell;
* Local Maxima in $A(\rho, \theta)$ correspond to the most probably lines in image space.

#### Edge Detection
The first step of the Hough Transform is given by the **Edge Detection**.
The most advanced Edge Detector we consider is the **Canny Edge Detector**: we will use the Opencv "cv2.Canny" function and will tune its parameters.


First of all we smooth the image by applying a **Gaussian filter**, which can be made in an adaptive way as follows:

In [ ]:
# FUNCTION THAT APPLIES GAUSSIAN BLUR TO AN IMAGE BY (ONLINE) CHANGING  OF THE SIGMA PARAMETER
def GaussianSmooth(img, sigma=1):
    window_name = "Gaussian Blur - Sigma Tuning"
    cv.namedWindow(window_name)
    
    # Trackbar: sigma * 10 (we start from 1 and we can change up to 5 -> we will see 10->50)
    sigma_x10=int(sigma*10)
    cv.createTrackbar("sigma x10", window_name, sigma_x10, 50, lambda x: None)
    
    while True:
        # get sigma by trackbar
        sigma = cv.getTrackbarPos("sigma x10", window_name)
        sigma = max(sigma / 10.0, 0.1)   # to avoid issue with the zero (min value = 0.1)
        
        # Applica Gaussian blur
        blurred_img = cv.GaussianBlur(img, (0, 0), sigma)
        
        # Mostra immagine con sigma sovrapposto
       # display_img = blurred_img.copy()
        
        #cv.putText(display_img, f"Sigma: {sigma:.1f}", (10, 30),
                   #cv.FONT_HERSHEY_SIMPLEX, 1, (255), 2)
        cv.imshow(window_name, blurred_img)
        
        #  Exit with "q" and print the final sigma value
        if cv.waitKey(1) & 0xFF == ord('q'):
            print('--- Gaussian Blur Parameters: ---')
            print(f'sigma: {sigma}')
            break

    cv.destroyAllWindows()
    return blurred_img, sigma

Now we want to apply the **Canny Edge Detection function** adding the possibility of changing the result by let varying the **Thresholds parameters $T_l$ and $T_H$**.

Recall that:
* The samples greater or equal to $T_H$ will be considered as edges;
* The samples with value between $T_l$ and $T_H$ will be considered as potential edges, meaning that they will be labeled as edges if it is possible to reach a pixel with Intensity greater or equal to $T_H$ by following a continuous path of pixels with value greater or equal to $T_L$.

**Observation:** the thresholds are in the interval $(0, 255)$ since the images are 8-bits ones:

In [ ]:
img = cv.imread("Images/road1b.png")
print(img.dtype, img.max(), img.min())

In [ ]:
# FUNCTION THAT EXECUTES THE CANNY EDGE DETECTION BY (ONLINE) CHANGING OF THE THRESHOLDS PARAMETERS
def Canny(img, T_l=50, T_H=150):
    cv.namedWindow('Canny')

    # TRACKBARS:
    cv.createTrackbar('Low thresh.', 'Canny', T_l, 255, lambda x: None)  # init T_l = 50, maximum T_l = 255
    cv.createTrackbar('High thresh.', 'Canny', T_H, 255, lambda x: None) # init T_H = 150, maximum T_H = 255
    
    while True:
        T_l = cv.getTrackbarPos('Low thresh.', 'Canny')
        T_H = cv.getTrackbarPos('High thresh.', 'Canny')
        
        edges = cv.Canny(img, threshold1=T_l, threshold2=T_H)  # APPLY CANNY EDGE DETECTION
        cv.imshow('Canny', edges)
        
        #  Exit with "q" and print the final Thresholds
        if cv.waitKey(1) & 0xFF == ord('q'):
            print('--- Canny Parameters: ---')
            print(f'Final Low threshold: {T_l}')
            print(f'Final High threshold: {T_H}')
            break
    cv.destroyAllWindows()
    return edges, T_l, T_H

#### Street Line Detection
Regarding the last three steps of the Hough Transform we can exploit one of the already implemented OpenCV functions. Also in this case we want to tune the result by letting vary the **parameters driving the Line Detection**.
They are:
* $\rho$ which represents the **"distance" resolution of the accumulator in pixels** for its quantization;
* $\theta$ which represents the **angle resolution of the accumulator in radians** for its quantization;
* **Quantized interval** $[\theta_{\min}, \theta_{\max}]$ in which $\theta$ can varyies and, as consequence, also $\rho = x\cos \theta+y\sin \theta$;
* $T_A$ which is the **Accumulator Threshold** $A(\rho, \theta)$ for the counters such that a line is detected.

**Observation:** The function will return the couple $(\rho, \theta$) corresponding to the detected lines in the image space. It means that **we also need to implement a part that actually draw the lines!**


In particular, the latter can be implemented as follows:

In [ ]:
def draw_lines(img: np.ndarray, lines: np.ndarray, colour: tp.List[int] = [0, 0, 255], thickness: int = 2) -> tp.Tuple[np.ndarray, np.ndarray, list]:
   
    # want to work on colour images (to get colour lines)
    if len(img.shape) == 2:
        output_img = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    else:
        # Se è già a colori, ne facciamo una copia per non rovinare l'originale
        output_img = img.copy()

    # Se non ci sono linee, restituisci tutto vuoto
    if lines is None:
        return output_img
    
    for rho, theta in lines:
        # extracts the cartesian quatntities associated to the polar ones
        x, y = polar2cartesian(rho, theta)
        
        # Extreme Points to draw the lines in in the image
        pt1 = np.round(x + 10000*y).astype(int)
        pt2 = np.round(x - 10000*y).astype(int)
        
        # Draw
        cv.line(output_img, pt1, pt2, [0,255,0], thickness)

    return output_img

While the actual **Line Detection** is computed, also in this case in an adaptive way, by the function:

In [ ]:
# FUNCTION TO DETECT STREET LINES WITH HOUGH TRANSFORM BY ONLINE CHANGING ITS PARAMETERS
def StreetDetection(edges, rho=1, theta_deg=1, A_T=50, min_th=0, max_th=180):
    
    cv.namedWindow('Lines Detection')
    cv.createTrackbar('Rho_res [px]', 'Lines Detection', rho, 10, lambda x: None)  # init rho_res = 1 -> max = 10
    cv.createTrackbar('Theta_res [deg]', 'Lines Detection', theta_deg, 10, lambda x: None) # init theta_res = 1° -> max = 10° (bad)
    cv.createTrackbar('Acc threshold', 'Lines Detection', A_T, 300, lambda x: None) # init A_T = 50 -> max = 300
    cv.createTrackbar('Min theta [deg]', 'Lines Detection', min_th, 180, lambda x: None)  # init min_theta = 0°  -> max = 180°
    cv.createTrackbar('Max theta [deg]', 'Lines Detection', max_th, 180, lambda x: None)  # init max_theta = 180° -> max = 180°

    
    while True:
        rho = max(1, cv.getTrackbarPos('Rho_res [px]', 'Lines Detection'))
        
        theta_deg = max(1, cv.getTrackbarPos('Theta_res [deg]', 'Lines Detection'))
        theta = theta_deg * np.pi / 180
        
        A_T = cv.getTrackbarPos('Acc threshold', 'Lines Detection')
        
        min_theta = cv.getTrackbarPos('Min theta [deg]', 'Lines Detection')
        min_th_rad = min_theta*(np.pi/180)
        max_theta = cv.getTrackbarPos('Max theta [deg]', 'Lines Detection')
        max_th_rad = max_theta*(np.pi/180)
    
        # To Keep max theta >= min theta
        if max_theta <= min_theta:
            max_theta = min_theta + (np.pi / 180)
    
        # Hough Transform
        lines = cv.HoughLines(
            edges,
            rho=rho,
            theta=theta,
            threshold=A_T,
            min_theta=min_th_rad,
            max_theta=max_th_rad
        )
    
        # Lines Draw
        if lines is not None:
            lines = lines.squeeze(1)
            img_lines = draw_lines(edges, lines, colour=[0,255,0], thickness=2)
        else:
            img_lines = edges.copy()

        cv.imshow("Lines Detection", img_lines)
    
        # Exit
        if cv.waitKey(1) & 0xFF == ord('q'):
            # 
            Hough_params = {
                "rho": rho,
                "theta_deg": theta_deg,
                "A_T": A_T,
                "min_th": min_theta,
                "max_th": max_theta
            }
            
            print('--- Hough transform parameters: ---')
            for key, value in Hough_params.items():
                print(f'  - {key}: {value}')
                
            cv.destroyAllWindows()
            break
    return img_lines, lines, Hough_params

### Road Lanes Detection

In order to achieve **road lanes detection**, we can exploit a different approach.
In particular we can think to use the HougLinesP() Opencv function which implement the probabilistic Hough Transform and returns the coordinates of segments extremes in the image space. This time the parameters we should tune are:
* $\rho$ resolution, as before;
* $\theta$ resolution, as before;
* Accumulator threshold $A_T$, as before;
* The minimum length of a segment;
* The maximum distance between points on the same line to link them.

However, I chosed to **add four parameters** to be tuned:
* One is the **Solid Threshold** in order to detect a line to be "dash" or "solid";
* The second and third one are the **Slope Threshold** and **Distance Threshold**. They will be used to group up lines that are very similar in terms of slope and, at the same time, very near in terms of euclidian distance.
* The last parameter is the **Horizon Ratio**. It allows us to extend the lines towards the horizon. This is very helpful when we detect small lines (like in the case of dash ones).

These parameters will be used by exploiting **some auxiliary functions** that I've chosed to implement in order to help us in the detection of the road lanes.
They are:

1) The **first function** that we will see allows us to **classify the lines**: they are **solid** or **dashed** based on the lines length detected by the Probabilistic Hough Transform. Indeed, ideally, a detected dashed line should be much shorter than a solid one.

In [ ]:
def classifyLines(lines, solid_thresh):

    solid_lines = []
    dashed_lines = []
    
    for line in lines:
        x1, y1, x2, y2 = line[0]
        length = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)  # length of the detected line
        
        if x2 != x1:  # i.e. if the line is not vertical
            slope = (y2-y1)/(x2-x1) 
            if abs(slope) < 0.3: continue   # basically we filter out all the quasi-horizontal lines
        
        if length > solid_thresh:  # solid line is detected!
            solid_lines.append(line)  # update the list
        else:  # dashed line is detected
            dashed_lines.append(line)  # update the list
            
    return solid_lines, dashed_lines

2) The **second one** allows us to obtain a **single line when we detect a group of nearby lines**. Its implementation is the following one:

In [ ]:
def merge_lines_segment(lines, slope_thresh=0.1, dist_thresh=20):
    
    if not lines:
        return []

    # Computation of slope and intercept
    lines_params = []  # we will store the semgnets extremes coordinates, the segment slope and intercent.
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x2 == x1: # manage of vertical lines (infinity slope)
            slope = 999.0
            intercept = x1
        else:
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - slope * x1
        lines_params.append({'coords': [x1, y1, x2, y2], 'm': slope, 'q': intercept}) # append

    merged_lines = []
    used_indices = set()

    # Clustering-like Algorithm
    for i in range(len(lines_params)):
        if i in used_indices:
            continue  # to avoid double lines

        base = lines_params[i]
        group = [base['coords']]
        used_indices.add(i)

        for j in range(i + 1, len(lines_params)):
            if j in used_indices:
                continue
            
            other = lines_params[j]
            # Similarity check in terms of "similar slope" and "distance"
            if abs(base['m'] - other['m']) < slope_thresh and abs(base['q'] - other['q']) < dist_thresh:
                group.append(other['coords'])
                used_indices.add(j)

        # Let's make the average to get the resulting merged lines
        group_arr = np.array(group)
        avg_coords = np.mean(group_arr, axis=0, dtype=int)
        merged_lines.append([avg_coords])

    return merged_lines

3) The **third axuiliary function** helps us to **extend the lines towards the horizon bases on a "top ratio"**. In particular, given the lines detected by the Hough Transform it returns a list containing the extreme coordinates of the (new) extendend segments

In [ ]:
def extrapolate_lines(lines, img_shape, top_ratio=0.6):
    
    if not lines:
        return []

    height, width = img_shape[:2]
    # definition of the y-limits: from the bottom to the horizon
    y_bottom = height
    y_top = int(height * top_ratio)

    # list containing the extended lines
    extended_lines = []

    for line in lines:
        x1, y1, x2, y2 = line[0]

        # Avoid vertical lines
        if x2 == x1:
            continue
            
        # Compute slope and bias of the line:
        m = (y2 - y1) / (x2 - x1)
        q = y1 - m * x1

        # Neglect the quasi-horizontal lines:
        if abs(m) < 0.1:
            continue

        # corresponging x = (y-q)/m
        x_bottom = int((y_bottom - q) / m)
        x_top = int((y_top - q) / m)

        # Update the list
        extended_lines.append([[x_bottom, y_bottom, x_top, y_top]])

    return extended_lines

4) The **fourth function** allows us to **draw the road lanes** in the given image by dinstinguish if they are solid or dashed:

In [ ]:
# FUNCTION THAT DRAW THE LINES IN THE STREET TO IDENTIFY THE ROAD LANES
def draw_lanes(img, solid_lines, dashed_lines, solid_color=(0, 0, 255), dashed_color=(0, 255, 0), thickness=2):
 
    # want to work on colour images
    if len(img.shape) == 2:
        output_img = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    else:
        output_img = img.copy()

    #  --- DRAW OF SOLID LINES ---
    if solid_lines is not None:
        for line in solid_lines:
            # HoughLinesP restituisce un array annidato [[x1, y1, x2, y2]]
            x1, y1, x2, y2 = line[0]
            cv.line(output_img, (x1, y1), (x2, y2), solid_color, thickness)

    #  --- DRAW OF DASHED LINES ---
    if dashed_lines is not None:
        for line in dashed_lines:
            x1, y1, x2, y2 = line[0]
            cv.line(output_img, (x1, y1), (x2, y2), dashed_color, thickness)

    return output_img

5) The **fifth (and last) auxiliary function** allows us to also **fill the area between detected (and drawn) lanes to completely identify the Road Lanes**. The function is based on:
   * solid-solid -> no need to be fill because it's not a road lane;
   * solid-dash  -> need to be fill because we have a road lane;
   * dash-solid  -> need to be fill;
   * dash-dash   -> need to be fill because it is one of the "central" road lanes of the street.

In [ ]:
# FUNCTION THAT FILL THE ROAD LANES
def fill_lanes(img, solid_lines, dashed_lines, color=(0, 255, 255), alpha=0.3):

    overlay = img.copy()

    # list of all the lines
    all_lines = []
    
    if solid_lines:
        for line in solid_lines:
            all_lines.append({'coords': line[0], 'type': 'solid'})   # store the coordinates of the solid line
            
    if dashed_lines:
        for line in dashed_lines:
            all_lines.append({'coords': line[0], 'type': 'dashed'})  # store the coordinates of the dashed line
    
    # Skip the function if we don't have at least two lines (no lane)
    if len(all_lines) < 2:
        return img

    # SORT OF THE LINES: FROM LEFT -> RIGHT based on x coordinate
    all_lines.sort(key=lambda item: item['coords'][0])

    # Iterate over all the adjacent lines (due to the previous sort)
    for i in range(len(all_lines) - 1):
        left_line = all_lines[i]  # this is the "left line"
        right_line = all_lines[i+1]  # this is the "right line"
        
        # Extract the coordinates: [x_bottom, y_bottom, x_top, y_top] (based on Horizon value of previous fnc)
        l_x_bot, l_y_bot, l_x_top, l_y_top = left_line['coords']
        r_x_bot, r_y_bot, r_x_top, r_y_top = right_line['coords']
        
        t1 = left_line['type']  # solid/dashed
        t2 = right_line['type'] # solid/dashed
        
        # Skip the coloring if the adjacent lines are both solid:
        if t1 == 'solid' and t2 == 'solid':
            continue
            
        # HERE WE HAVE ONE OF THE VALID COMBINATIONS: (Solid-Dashed, Dashed-Solid, or Dashed-Dashed)
        
        # definition of the polygon points to be coloured (order: Bottom-Left -> Top-left -> Top-right -> Bottom-right)
        pts = np.array([
            [l_x_bot, l_y_bot], # Bottom Left
            [l_x_top, l_y_top], # Top Left
            [r_x_top, r_y_top], # Top Right
            [r_x_bot, r_y_bot]  # Bottom Right
        ], np.int32)
        
        # Reshape for the fillPoly function
        pts = pts.reshape((-1, 1, 2))
        # Draw and colour the polygon on the overlay image
        cv.fillPoly(overlay, [pts], color)

    # Combine the "new image" overlay with the original one with a weighted sum driven by alpha
    final_img = cv.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    return final_img

Let's now see the implementation of our big **Road Detection Function**. It puts everything we mentioned above:

In [ ]:
def RoadDetection(edges, rho=1, theta=1, A_T=50, min_Len=20, max_Gap=10, solid_Th=60, merge_slope=10, merge_dist=10, horizon_ratio=60):
    
    window_name = 'Road Detection Tuning'
    cv.namedWindow(window_name)

    # ---- MANAGE OF NOT INTEGER VALUES FOR THE TRACKBAR ----
    merge_slope = int(merge_slope * 100) if merge_slope <= 1.0 else int(merge_slope)
    horizon_ratio = int(horizon_ratio * 100) if horizon_ratio <= 1.0 else int(horizon_ratio)
    theta = int(np.degrees(theta)) if theta < 1.0 else int(theta)  # in case of radiant theta input
    # -----------------------------------------------------------------------------

    # --- TRACKBARS ---
    cv.createTrackbar('Rho [px]', window_name, rho, 10, lambda x: None)
    cv.createTrackbar('Theta [deg]', window_name, theta, 10, lambda x: None)
    cv.createTrackbar('Acc Threshold', window_name, A_T, 450, lambda x: None)
    cv.createTrackbar('Min Len [px]', window_name, min_Len, 200, lambda x: None)
    cv.createTrackbar('Max Gap [px]', window_name, max_Gap, 100, lambda x: None)
    
    cv.createTrackbar('Solid Thresh [px]', window_name, solid_Th, 300, lambda x: None)
    
    # Ora merge_slope e horizon_ratio sono sicuramente INT corretti (es. 20 e 60)
    cv.createTrackbar('Merge Slope', window_name, merge_slope, 100, lambda x: None) 
    cv.createTrackbar('Merge Dist [px]', window_name, merge_dist, 50, lambda x: None)
    cv.createTrackbar('Horizon %', window_name, horizon_ratio, 90, lambda x: None)

    while True:
        rho = max(1, cv.getTrackbarPos('Rho [px]', window_name))
        theta = max(1, cv.getTrackbarPos('Theta [deg]', window_name))
        theta_rad = np.deg2rad(theta)
        
        Acc_T = cv.getTrackbarPos('Acc Threshold', window_name)
        min_len = cv.getTrackbarPos('Min Len [px]', window_name)
        max_gap = cv.getTrackbarPos('Max Gap [px]', window_name)
        solid_thresh = cv.getTrackbarPos('Solid Thresh [px]', window_name)
        
        slope_thresh = cv.getTrackbarPos('Merge Slope', window_name) / 100.0 
        dist_thresh = cv.getTrackbarPos('Merge Dist [px]', window_name)
        horizon_val = cv.getTrackbarPos('Horizon %', window_name) / 100.0

        # colour image to draw colours lines
        display_img = cv.cvtColor(edges, cv.COLOR_GRAY2BGR)
        
        # --- PROBABILISTIC HOUGH TRANSFORM ---
        lines = cv.HoughLinesP(edges, rho, theta_rad, Acc_T, minLineLength=min_len, maxLineGap=max_gap)

        final_solid = []
        final_dashed = []

        if lines is not None:
            # --- SOLID/DASHED LINE CLASSIFICATION ---
            solid_lines, dashed_lines = classifyLines(lines, solid_thresh)

            # --- MERGE OF "SIMILAR" LINES ---
            merged_solid = merge_lines_segment(solid_lines, slope_thresh, dist_thresh)
            merged_dashed = merge_lines_segment(dashed_lines, slope_thresh, dist_thresh)

            # --- EXTENSION OF THE LINES ---
            final_solid = extrapolate_lines(merged_solid, edges.shape, top_ratio=horizon_val)
            final_dashed = extrapolate_lines(merged_dashed, edges.shape, top_ratio=horizon_val)

            # --- LANES DETECTION BY DRAWING AND FILLING THEM ---
            display_img = draw_lanes(display_img, final_solid, final_dashed)
            display_img = fill_lanes(display_img, final_solid, final_dashed, color=(0, 255, 255), alpha=0.4)
            
        # --- SHOW THE CURRENT RESULT ---
        cv.imshow(window_name, display_img)

        # Exit by pressing "q" and see the final (tuned) parameters
        if cv.waitKey(1) & 0xFF == ord('q'):
            # Costruzione del Dizionario
            RoadDet_params = {
                "rho": rho,
                "theta": theta_rad,   
                "A_T": Acc_T,
                "min_Len": min_len,
                "max_Gap": max_gap,
                "solid_Th": solid_thresh,
                "merge_slope": slope_thresh,
                "merge_dist": dist_thresh,
                "horizon_ratio": horizon_val
            }

            print("--- Road Lanes Detection Parameters: ---")
            for k, v in RoadDet_params.items():
                print(f"  - {k}: {v}")
            cv.destroyWindow(window_name)
            
            return display_img, final_solid, final_dashed, RoadDet_params

### Results [Simplified Images]

Let's now show what we've implemented above by using, as images, all the simplified ones that we have in our dataset...

* FUNCTION RUNNING THE ROAD DETECTION PIPELINE:

In [ ]:
def RoadDetPipeline(img_path, my_metadata=None):  # my_metadata will contain some loaded data we want to use

    # --- LOADING IMAGE ---
    img = cv.imread(img_path)
    if img is None:
        print("Image not found!")
        return None

    base_name = os.path.splitext(os.path.basename(img_path))[0]

    print(f"PROCESSING IMAGE: {base_name}...")
    print("\n")

    gauss_kwargs = {}
    canny_kwargs = {}
    street_kwargs = {}
    road_kwargs = {}

    if my_metadata:
        gauss_kwargs["sigma"] = my_metadata["Gaussian_Sigma"]
        canny_kwargs["T_l"] = my_metadata["Canny_Threshold_Low"]
        canny_kwargs["T_H"] = my_metadata["Canny_Threshold_High"]
        for key, value in my_metadata.items():
            # Se la chiave inizia con "Street_", la puliamo e la mettiamo in street_kwargs
            if key.startswith("Street_"):
                param_name = key[7:]   # remove "Street_"
                street_kwargs[param_name] = value
            elif key.startswith("Road_"):
                param_name = key[5:]  # remove "Road_"
                road_kwargs[param_name] = value

    # --- PIPELINE --- 
    # Gaussian Filter
    blurred_img, sigma = GaussianSmooth(img.copy(), **gauss_kwargs) 
    print("\n")
    # Canny edge detection
    Canny_img, T_l, T_H = Canny(blurred_img, **canny_kwargs)
    print("\n")
    # street detection
    _, street_lines, Street_params = StreetDetection(Canny_img, **street_kwargs)
    print("\n")
    street_detected_img = draw_lines(img.copy(), street_lines)  # draw of the detected street lines on the original img
    # road detection
    _, solid_lines, dashed_lines, RoadDet_params = RoadDetection(Canny_img, **road_kwargs)
    print("\n")
    road_detected_img = draw_lanes(img.copy(), solid_lines, dashed_lines) # draw of the detected road lanes on original img
    road_detected_img = fill_lanes(road_detected_img, solid_lines, dashed_lines, color=(0, 255, 255), alpha=0.3) # fill also the lanes

    # --- PIPELINE STEPS ---
    steps = [
        ("0_Original", img),
        ("1_Gaussian", blurred_img),
        ("2_Canny", Canny_img),
        ("3_StreetDet", street_detected_img),
        ("4_RoadDet", road_detected_img)
    ]

        
    # --- METADATA ---
    metadata = {
        "Image_ID": base_name,
        "Original_Path": img_path,
        "Gaussian_Sigma": float(sigma),
        "Canny_Threshold_Low": int(T_l),
        "Canny_Threshold_High": int(T_H),
        **{f"Street_{k}": v for k, v in Street_params.items()},
        **{f"Road_{k}": v for k, v in RoadDet_params.items()}
    }

    # --- RESULTS ---
    results = {
        "id": base_name,
        "metadata": metadata,
        "images": steps
    }
    
    print("-" * 30)
    print("\n")
    
    return results

* FUNCTION STORING THE RESULTS

In [ ]:
def SaveDATA(data, folder, filename):
    """
    Save results in pickle and save images as PNGs.
    Works with a single result (dict) or a list of results.
    """
    if not os.path.exists(folder):
        os.makedirs(folder)

    # salva pickle
    path = os.path.join(folder, filename)
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f"Results stored successfully in: {path}")

    # normalizza in lista
    data_list = data if isinstance(data, list) else [data]

    # salva immagini
    for item in data_list:
        img_folder = os.path.join(folder, item["id"])
        os.makedirs(img_folder, exist_ok=True)
        for name, img in item["images"]:
            img_path = os.path.join(img_folder, f"{name}.png")
            cv.imwrite(img_path, img)
        print(f"Images saved in: {img_folder}")


* FUNCTION LOADING THE RESULTS

In [ ]:
def LoadDATA(folder, filename):
    full_path = os.path.join(folder, filename)

    if os.path.exists(full_path):
        with open(full_path, 'rb') as f:
            return pickle.load(f)
    else:
        print("Error: File does not exist.")
        return None

* FUNCTION SHOWING THE ROAD DETECTION RESULTS

In [ ]:
def showResults(results_list, titles=None):
    
    for i, row in enumerate(results_list):
        cols = len(row)
        scale_w = 3
        scale_h = 2
        fig, axes = plt.subplots(1, cols, figsize=(cols*scale_w, scale_h))

        if cols == 1:
            axes = [axes]

        for j in range(cols):
            img = row[j]
            ax = axes[j]

            if len(img.shape) == 2:
                ax.imshow(img, cmap='gray')  # manage of grey-scale images
            else:
                ax.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))  # manage of BGR images (for matplot)

            ax.axis('off')

            if titles is not None:
                ax.set_title(titles[j], fontsize=12, pad=2)

        
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0,
                            wspace=0.01)
        plt.show()

* RESULTS: Let's see the pipeline in action

In [ ]:
simp_images_path = ['Images/road1b.png', 'Images/road2b.jpg', 'Images/road3b.jpg', 'Images/road4b.png']
simp_path = "DATA/Simp_Data" 

results = []
for path in simp_images_path:
    img_id = os.path.splitext(os.path.basename(path))[0]
    # PRESS ONLY "q"!
    results.append(RoadDetPipeline(path))

SaveDATA(results, simp_path, file_name)

In [ ]:
titles = ["Original", "Gaussian Smoothing", "Canny Edge Detection", "Street Lines Detection", "Road Detection"]
images = [ [step[1] for step in item["images"]] for item in results ]
showResults(images, titles)

---

## Street Lines and Road Detection [Full Version]

Now we want to deal with the original images. They differ from the previous ones by the fact that the street was already extracted. It means that, before running the Pipeline we have just implemented, we need to compute some pre-process computations to extract the "so-called" ROI (Region of Interest), which is precisely the street.

### ROI Extraction and Complete Road Detection

Regarding the ROI estraction, it is convenient to implement a function that, in adaptive way, like the previous functions, allows us to tune some parameters to choose what is the region of interest: in this case the street.
In particular, our idea is to build a rectangular from the bottom of the image and then attach a "dynamic" trapeizoidal to the rectangular looking toward the vanishing point.

Let'see my implementation:

In [ ]:
def ROIextraction(img):
    
    h, w = img.shape[:2]  # extract height and width of the image
    window = "ROI Tuning"
    cv.namedWindow(window)

    # --- TRACKBAR ---
    cv.createTrackbar("Rect Height", window, int(h*0.4), h, lambda x: None)  # Height of Rectangular 
    cv.createTrackbar("Top Width", window, int(w*0.2), w, lambda x: None)  # TOP width
    cv.createTrackbar("Trap Height", window, int(h*0.3), h, lambda x: None) # Height of Trapezoidal
    cv.createTrackbar("Top Shift", window, w//2, w, lambda x: None)  # Top shift to move the vanishing point (punto di fuga)

    while True:
        rect_h = cv.getTrackbarPos("Rect Height", window)
        top_w  = cv.getTrackbarPos("Top Width", window)
        trap_h = cv.getTrackbarPos("Trap Height", window)
        top_shift = cv.getTrackbarPos("Top Shift", window) - w//2

        y_bottom = h
        y_rect_top = max(0, h - rect_h)
        y_trap_top = max(0, y_rect_top - trap_h)

        # Rettangolo sempre centrato e largo quanto l'immagine
        xc_bottom = w // 2
        rect_w = w

        # lateral vanishing point
        xc_top = xc_bottom + top_shift

        # Rectangular points (fixed ones)
        bl = (0, y_bottom)
        br = (w, y_bottom)
        ml = (0, y_rect_top)
        mr = (w, y_rect_top)

        # TOP Trapezoidal points
        tl = (int(xc_top - top_w/2), y_trap_top)
        tr = (int(xc_top + top_w/2), y_trap_top)

        # Polygonal points
        pts = np.array([[bl, br, mr, tr, tl, ml]], dtype=np.int32)

        mask = np.zeros_like(img)
        cv.fillPoly(mask, pts, (255,255,255))

        # --- GET THE ROI ---
        ROI = cv.bitwise_and(img, mask)

        
        if len(img.shape) == 2:
            image = cv.cvtColor(img, cv.COLOR_GRAY2BGR)  # CONVERT TO BGR TO COLOUR THE POLYGON
        else:
            image = img.copy()
        
        # Overlay for the polygon:
        overlay = image.copy()
        
        # Filling the polygon 
        cv.fillPoly(overlay, pts, (255, 0, 0))  # BGR
        cv.polylines(overlay, pts, True, (0, 255, 255), 4)
        
        # GET THE POLYGON ON THE GIVEN IMAGE
        alpha = 0.35
        image = cv.addWeighted(overlay, alpha, image, 1 - alpha, 0)
        
        cv.imshow(window, image)


        if cv.waitKey(1) & 0xFF == ord('q'):
            print("--- ROI successfully extracted. ---\n")
            break

    cv.destroyWindow(window)
    return ROI, pts

In [ ]:
# example of use:
img = cv.imread("Images/road1.png")
roi_img, pts = ROIextraction(Canny(GaussianSmooth(img)[0])[0])
show_img(roi_img)

It basically means that we can exploit the RoadDetPipeline() function by just adding a very small change: the ROI extraction, if needed. Actually, I've seen that **we can get better performances on the Road Detection if we extract The Region Of Interest after the Edge Detection**.

Let's see the **Full Road Detection function**:

In [ ]:
def RoadDetPipeline(img_path, my_metadata=None, ROI: bool=True):  # my_metadata will contain some loaded data we want to use

    # --- LOADING IMAGE ---
    img = cv.imread(img_path)
    if img is None:
        print("Image not found!")
        return None

    base_name = os.path.splitext(os.path.basename(img_path))[0]

    print(f"PROCESSING IMAGE: {base_name}...")
    print("\n")

    gauss_kwargs = {}
    canny_kwargs = {}
    street_kwargs = {}
    road_kwargs = {}

    roi_pts = None

    if my_metadata:
        gauss_kwargs["sigma"] = my_metadata["Gaussian_Sigma"]
        canny_kwargs["T_l"] = my_metadata["Canny_Threshold_Low"]
        canny_kwargs["T_H"] = my_metadata["Canny_Threshold_High"]
        if "ROI_Parameters" in my_metadata and my_metadata["ROI_Parameters"] is not None:
            roi_pts = np.array(my_metadata["ROI_Parameters"], dtype=np.int32)
            
        for key, value in my_metadata.items():
            # Se la chiave inizia con "Street_", la puliamo e la mettiamo in street_kwargs
            if key.startswith("Street_"):
                param_name = key[7:]   # remove "Street_"
                street_kwargs[param_name] = value
            elif key.startswith("Road_"):
                param_name = key[5:]  # remove "Road_"
                road_kwargs[param_name] = value

        
    # --- PIPELINE --- 
    
    # Gaussian Filter
    blurred_img, sigma = GaussianSmooth(img.copy(), **gauss_kwargs) 
    print("\n")
    # Canny edge detection
    Canny_img, T_l, T_H = Canny(blurred_img, **canny_kwargs)
    print("\n")
    Canny_original = Canny_img.copy()
    # ROI extraction (eventually)
    if ROI:
        if roi_pts is not None:  # we make the ROI extraction automatically
            mask = np.zeros_like(Canny_img)
            cv.fillPoly(mask, [roi_pts], 255)
            Canny_img = cv.bitwise_and(Canny_img, mask)
        else:  # need to tune the ROI extractor
            Canny_img, roi_pts = ROIextraction(Canny_img)
    # street detection
    _, street_lines, Street_params = StreetDetection(Canny_img, **street_kwargs)
    print("\n")
    street_detected_img = draw_lines(img.copy(), street_lines)  # draw of the detected street lines on the original img
    # road detection
    _, solid_lines, dashed_lines, RoadDet_params = RoadDetection(Canny_img, **road_kwargs)
    print("\n")
    road_detected_img = draw_lanes(img.copy(), solid_lines, dashed_lines) # draw of the detected road lanes on original img
    road_detected_img = fill_lanes(road_detected_img, solid_lines, dashed_lines, color=(0, 255, 255), alpha=0.3) # fill also the lanes

    
    # --- PIPELINE STEPS ---
    steps = [
        ("0_Original", img),
        ("1_Gaussian", blurred_img),
        ("2_Canny", Canny_original),
        ("3_StreetDet", street_detected_img),
        ("4_RoadDet", road_detected_img)
    ]
    if ROI:
        steps.insert(3, ("2B_ROIextraction", Canny_img))

        
    # --- METADATA ---
    metadata = {
        "Image_ID": base_name,
        "Original_Path": img_path,
        "Gaussian_Sigma": float(sigma),
        "Canny_Threshold_Low": int(T_l),
        "Canny_Threshold_High": int(T_H),
        **({"ROI_Parameters": roi_pts} if ROI else {}), 
        **{f"Street_{k}": v for k, v in Street_params.items()},
        **{f"Road_{k}": v for k, v in RoadDet_params.items()}
    }

    # --- RESULTS ---
    results = {
        "id": base_name,
        "metadata": metadata,
        "images": steps
    }
    
    print("-" * 30)
    print("\n")
    
    return results

### Results [Original images]

Let's see the entire pipeline in action:

In [ ]:
images_path = ['Images/road1.png', 'Images/road2.jpg', 'Images/road3.jpg', 'Images/road4.png']
data_path = "DATA/original_DATA" # data in "DATA" folder may be overwritten. Use "Best_DATA" to restore my parameters.

results = []
for path in images_path:
    img_id = os.path.splitext(os.path.basename(path))[0]
    # PRESS ONLY "q" to see the results.
    results.append(RoadDetPipeline(path, ROI=True))
    
# SaveDATA(results, data_path, file_name)  # SCOMMENT TO SAVE THE NEW RESULTS IN FOLDER

In [ ]:
# Show the results:
titles = ["Original", "Gaussian Smoothing", "Canny Edge Detection", "ROI Extraction", "Street Lines Detection", "Road Detection"]
images = [ [step[1] for step in item["images"]] for item in results ]
showResults(images, titles)